In [1]:
import pandas as pd
import numpy as np

In [39]:
"""
F1 Race Weather Enrichment
--------------------------
Adds 'rain' and 'sunny' binary columns to races.csv using Open-Meteo historical API.
- rain  = 1 if precipitation > 0.1mm during any hour of the race window
- sunny = 1 if cloud cover < 65% during any hour of the race window
Both can be 1 (mixed), (0,0) is not allowed — defaults to sunny=1 if both are 0.

Usage:
    pip install requests pandas
    python add_weather.py
"""

import requests
import pandas as pd
import time

# ---------------------------------------------------------------------------
# Circuit coordinates  (lat, lon)
# ---------------------------------------------------------------------------
CIRCUIT_COORDS = {
    "Australian Grand Prix":        (-37.8497,  144.9680),
    "Bahrain Grand Prix":           ( 26.0325,   50.5106),
    "Chinese Grand Prix":           ( 31.3389,  121.2198),
    "Azerbaijan Grand Prix":        ( 40.3725,   49.8533),
    "Spanish Grand Prix":           ( 41.5700,    2.2611),
    "Monaco Grand Prix":            ( 43.7347,    7.4205),
    "Canadian Grand Prix":          ( 45.5000,  -73.5228),
    "French Grand Prix":            ( 43.2506,    5.7917),
    "Austrian Grand Prix":          ( 47.2197,   14.7647),
    "British Grand Prix":           ( 52.0786,   -1.0169),
    "German Grand Prix":            ( 49.3288,    8.5664),
    "Hungarian Grand Prix":         ( 47.5830,   19.2526),
    "Belgian Grand Prix":           ( 50.4372,    5.9714),
    "Italian Grand Prix":           ( 45.6156,    9.2811),
    "Singapore Grand Prix":         (  1.2914,  103.8640),
    "Russian Grand Prix":           ( 43.4057,   39.9578),
    "Japanese Grand Prix":          ( 34.8431,  136.5407),
    "Mexican Grand Prix":           ( 19.4042,  -99.0907),
    "Mexico City Grand Prix":       ( 19.4042,  -99.0907),
    "United States Grand Prix":     ( 30.1328,  -97.6411),
    "Brazilian Grand Prix":         (-23.7036,  -46.6997),
    "São Paulo Grand Prix":         (-23.7036,  -46.6997),
    "Abu Dhabi Grand Prix":         ( 24.4672,   54.6031),
    "Styrian Grand Prix":           ( 47.2197,   14.7647),   # same venue as Austrian
    "70th Anniversary Grand Prix":  ( 52.0786,   -1.0169),   # Silverstone
    "Tuscan Grand Prix":            ( 43.7275,   11.3711),   # Mugello
    "Eifel Grand Prix":             ( 50.3356,    6.9475),   # Nürburgring
    "Portuguese Grand Prix":        ( 38.4040,   -8.9902),   # Portimão
    "Emilia Romagna Grand Prix":    ( 44.3439,   11.7167),   # Imola
    "Turkish Grand Prix":           ( 40.9517,   29.4050),
    "Sakhir Grand Prix":            ( 26.0325,   50.5106),   # Bahrain outer loop
    "Dutch Grand Prix":             ( 52.3888,    4.5409),   # Zandvoort
    "Saudi Arabian Grand Prix":     ( 21.5433,   39.1728),
    "Miami Grand Prix":             ( 25.9581,  -80.2389),
    "Las Vegas Grand Prix":         ( 36.1147, -115.1728),
    "Qatar Grand Prix":             ( 25.4900,   51.4542),
}

# ---------------------------------------------------------------------------
# Thresholds
# ---------------------------------------------------------------------------
RAIN_THRESHOLD_MM   = 0.1   # mm/hr — light rain and above
SUNNY_CLOUD_MAX_PCT = 65    # % cloud cover below which it counts as sunny
RACE_DURATION_HRS   = 2     # window after race start to sample

# ---------------------------------------------------------------------------
# Fetch weather for one race
# ---------------------------------------------------------------------------
def fetch_weather(lat: float, lon: float, date: str, start_time: str, gp_name: str = "") -> dict:
    """
    Calls Open-Meteo archive API and returns rain/sunny flags.
    date: 'YYYY-MM-DD', start_time: 'HH:MM:SS' (UTC)
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": date,
        "end_date":   date,
        "hourly":     "precipitation,cloudcover",
        "timezone":   "UTC",
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    hours      = data["hourly"]["time"]           # list of 'YYYY-MM-DDTHH:00'
    precip     = data["hourly"]["precipitation"]  # mm/hr
    cloudcover = data["hourly"]["cloudcover"]     # %

    # Parse race start hour (UTC)
    start_hour = int(start_time.split(":")[0])
    end_hour   = min(start_hour + RACE_DURATION_HRS, 23)

    race_precip = [
        precip[i] for i, t in enumerate(hours)
        if start_hour <= int(t[11:13]) <= end_hour
    ]
    race_cloud = [
        cloudcover[i] for i, t in enumerate(hours)
        if start_hour <= int(t[11:13]) <= end_hour
    ]

    # Warn if API returned nulls in the race window
    null_precip = [i for i, p in enumerate(race_precip) if p is None]
    null_cloud  = [i for i, c in enumerate(race_cloud)  if c is None]
    if null_precip or null_cloud:
        print(f"    [WARN] {gp_name} {date} — nulls in race window")
        print(f"    [WARN] raw precip: {race_precip}")
        print(f"    [WARN] raw cloud:  {race_cloud}")

    rain  = int(any(p is not None and p > RAIN_THRESHOLD_MM for p in race_precip))
    sunny = int(any(c is not None and c < SUNNY_CLOUD_MAX_PCT for c in race_cloud))

    # Constraint: (0, 0) is not allowed — dry but overcast counts as sunny/normal
    # Also handles the edge case where all API values were None
    if rain == 0 and sunny == 0:
        sunny = 1

    return {"rain": rain, "sunny": sunny}


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    df = pd.read_csv("filler-datasets(kaggle)/races.csv")
    df = df[df["year"] >= 2019].copy().reset_index(drop=True)

    rain_col  = []
    sunny_col = []

    for _, row in df.iterrows():
        gp_name = row["name"]
        date    = row["date"]
        time_   = row["time"] if row["time"] != "\\N" else "13:00:00"

        if gp_name not in CIRCUIT_COORDS:
            print(f"  [SKIP] No coords for: {gp_name}")
            rain_col.append(None)
            sunny_col.append(None)
            continue

        lat, lon = CIRCUIT_COORDS[gp_name]
        try:
            result = fetch_weather(lat, lon, date, time_, gp_name)
            rain_col.append(result["rain"])
            sunny_col.append(result["sunny"])
            print(f"  [OK] {date} | {gp_name:35s} | rain={result['rain']} sunny={result['sunny']}")
        except Exception as e:
            print(f"  [ERR] {date} | {gp_name} | {e}")
            rain_col.append(None)
            sunny_col.append(None)

        time.sleep(0.3)   # be polite to the free API

    df["rain"]  = rain_col
    df["sunny"] = sunny_col

    out_path = "races_with_weather.csv"
    df.to_csv(out_path, index=False)
    print(f"\nDone! Saved to {out_path}")
    print(df[["raceId", "year", "name", "date", "rain", "sunny"]].to_string())


if __name__ == "__main__":
    main()

  [OK] 2019-03-17 | Australian Grand Prix               | rain=0 sunny=1
  [OK] 2019-03-31 | Bahrain Grand Prix                  | rain=0 sunny=1
  [OK] 2019-04-14 | Chinese Grand Prix                  | rain=0 sunny=1
  [OK] 2019-04-28 | Azerbaijan Grand Prix               | rain=0 sunny=1
  [OK] 2019-05-12 | Spanish Grand Prix                  | rain=0 sunny=1
  [OK] 2019-05-26 | Monaco Grand Prix                   | rain=1 sunny=0
  [OK] 2019-06-09 | Canadian Grand Prix                 | rain=0 sunny=1
  [OK] 2019-06-23 | French Grand Prix                   | rain=0 sunny=1
  [OK] 2019-06-30 | Austrian Grand Prix                 | rain=0 sunny=1
  [OK] 2019-07-14 | British Grand Prix                  | rain=1 sunny=1
  [OK] 2019-07-28 | German Grand Prix                   | rain=1 sunny=0
  [OK] 2019-08-04 | Hungarian Grand Prix                | rain=0 sunny=1
  [OK] 2019-09-01 | Belgian Grand Prix                  | rain=0 sunny=1
  [OK] 2019-09-08 | Italian Grand Prix             

In [40]:
df= pd.read_csv('races_with_weather.csv')
df = df.loc[:, ['raceId', 'year', 'name', 'rain', 'sunny']]
df[(df['rain'] == 1) & (df['sunny'] == 1)]

,raceId,year,name,rain,sunny
9,1019,2019,British Grand Prix,1,1
15,1025,2019,Russian Grand Prix,1,1
26,1036,2020,Spanish Grand Prix,1,1
46,1058,2021,Styrian Grand Prix,1,1
50,1062,2021,Hungarian Grand Prix,1,1
54,1066,2021,Russian Grand Prix,1,1
79,1094,2022,Mexico City Grand Prix,1,1
94,1111,2023,Dutch Grand Prix,1,1
112,1129,2024,Canadian Grand Prix,1,1
113,1130,2024,Spanish Grand Prix,1,1


In [41]:
df[df['name'].isin(['Russian Grand Prix', 'Belgian Grand Prix', 'German Grand Prix'])][['year', 'name', 'rain', 'sunny']]

,year,name,rain,sunny
10,2019,German Grand Prix,1,0
12,2019,Belgian Grand Prix,0,1
15,2019,Russian Grand Prix,1,1
27,2020,Belgian Grand Prix,1,0
30,2020,Russian Grand Prix,0,1
51,2021,Belgian Grand Prix,1,0
54,2021,Russian Grand Prix,1,1
73,2022,Belgian Grand Prix,0,1
93,2023,Belgian Grand Prix,0,1
117,2024,Belgian Grand Prix,0,1


In [48]:
main_race_res_2019 = pd.read_csv('main_race_result(processed)/formula1_2019season_raceResults.csv')
main_race_res_2020 = pd.read_csv('main_race_result(processed)/formula1_2020season_raceResults.csv')
main_race_res_2021 = pd.read_csv('main_race_result(processed)/formula1_2021season_raceResults.csv')
main_race_res_2022 = pd.read_csv('main_race_result(processed)/formula1_2022season_raceResults.csv')
main_race_res_2023 = pd.read_csv('main_race_result(processed)/formula1_2023season_raceResults.csv')
main_race_res_2024 = pd.read_csv('main_race_result(processed)/formula1_2024season_raceResults.csv')
main_race_res_2025 = pd.read_csv('main_race_result(processed)/formula1_2025season_raceResults.csv')

main_race_quali_2019 = pd.read_csv('quali_both_result(processed)/race_quali_2019.csv')
main_race_quali_2020 = pd.read_csv('quali_both_result(processed)/race_quali_2020.csv')
main_race_quali_2021 = pd.read_csv('quali_both_result(processed)/race_quali_2021.csv')
main_race_quali_2022 = pd.read_csv('quali_both_result(processed)/race_quali_2022.csv')
main_race_quali_2023 = pd.read_csv('quali_both_result(processed)/race_quali_2023.csv')
main_race_quali_2024 = pd.read_csv('quali_both_result(processed)/race_quali_2024.csv')
main_race_quali_2025 = pd.read_csv('quali_both_result(processed)/race_quali_2025.csv')

sprint_quali_2023 = pd.read_csv('quali_both_result(processed)/sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('quali_both_result(processed)/sprint_quali_2024.csv')
sprint_quali_2025 = pd.read_csv('quali_both_result(processed)/sprint_quali_2025.csv')

sprint_race_res_2021 = pd.read_csv('sprint_race_result(processed)/sprint_2021_result.csv')
sprint_race_res_2022 = pd.read_csv('sprint_race_result(processed)/sprint_2022_result.csv')
sprint_race_res_2023 = pd.read_csv('sprint_race_result(processed)/sprint_2023_result.csv')
sprint_race_res_2024 = pd.read_csv('sprint_race_result(processed)/sprint_2024_result.csv')
sprint_race_res_2025 = pd.read_csv('sprint_race_result(processed)/sprint_2025_result.csv')

In [49]:
weather_df = df[['raceId', 'rain', 'sunny']]
def add_weather(data):
    return data.merge(weather_df, on='raceId', how='left')

In [50]:
main_race_res_2019 = add_weather(main_race_res_2019)
main_race_res_2020 = add_weather(main_race_res_2020)
main_race_res_2021 = add_weather(main_race_res_2021)
main_race_res_2022 = add_weather(main_race_res_2022)
main_race_res_2023 = add_weather(main_race_res_2023)
main_race_res_2024 = add_weather(main_race_res_2024)


main_race_quali_2019 = add_weather(main_race_quali_2019)
main_race_quali_2020 = add_weather(main_race_quali_2020)
main_race_quali_2021 = add_weather(main_race_quali_2021)
main_race_quali_2022 = add_weather(main_race_quali_2022)
main_race_quali_2023 = add_weather(main_race_quali_2023)
main_race_quali_2024 = add_weather(main_race_quali_2024)


sprint_quali_2023 = add_weather(sprint_quali_2023)
sprint_quali_2024 = add_weather(sprint_quali_2024)


sprint_race_res_2021 = add_weather(sprint_race_res_2021)
sprint_race_res_2022 = add_weather(sprint_race_res_2022)
sprint_race_res_2023 = add_weather(sprint_race_res_2023)
sprint_race_res_2024 = add_weather(sprint_race_res_2024)


In [52]:
main_race_quali_2019.head()

,raceId,Team,kaggle_driver_id,position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec,Track,rain,sunny
0,1010,Mercedes,1,1,1,1,82.043,81.014,80.486,Australia,0,1
1,1010,Mercedes,822,2,1,1,82.367,81.193,80.598,Australia,0,1
2,1010,Ferrari,20,3,1,1,82.885,81.912,81.190,Australia,0,1
3,1010,Red Bull Racing,830,4,1,1,82.876,81.678,81.320,Australia,0,1
4,1010,Ferrari,844,5,1,1,82.017,81.739,81.442,Australia,0,1


In [53]:
main_race_res_2019.to_csv('main_race_result(processed)/formula1_2019season_raceResults.csv', index=False)
main_race_res_2020.to_csv('main_race_result(processed)/formula1_2020season_raceResults.csv', index=False)
main_race_res_2021.to_csv('main_race_result(processed)/formula1_2021season_raceResults.csv', index=False)
main_race_res_2022.to_csv('main_race_result(processed)/formula1_2022season_raceResults.csv', index=False)
main_race_res_2023.to_csv('main_race_result(processed)/formula1_2023season_raceResults.csv', index=False)
main_race_res_2024.to_csv('main_race_result(processed)/formula1_2024season_raceResults.csv', index=False)
main_race_res_2025.to_csv('main_race_result(processed)/formula1_2025season_raceResults.csv', index=False)

main_race_quali_2019.to_csv('quali_both_result(processed)/race_quali_2019.csv', index=False)
main_race_quali_2020.to_csv('quali_both_result(processed)/race_quali_2020.csv', index=False)
main_race_quali_2021.to_csv('quali_both_result(processed)/race_quali_2021.csv', index=False)
main_race_quali_2022.to_csv('quali_both_result(processed)/race_quali_2022.csv', index=False)
main_race_quali_2023.to_csv('quali_both_result(processed)/race_quali_2023.csv', index=False)
main_race_quali_2024.to_csv('quali_both_result(processed)/race_quali_2024.csv', index=False)
main_race_quali_2025.to_csv('quali_both_result(processed)/race_quali_2025.csv', index=False)

sprint_quali_2023.to_csv('quali_both_result(processed)/sprint_quali_2023.csv', index=False)
sprint_quali_2024.to_csv('quali_both_result(processed)/sprint_quali_2024.csv', index=False)
sprint_quali_2025.to_csv('quali_both_result(processed)/sprint_quali_2025.csv', index=False)

sprint_race_res_2021.to_csv('sprint_race_result(processed)/sprint_2021_result.csv', index=False)
sprint_race_res_2022.to_csv('sprint_race_result(processed)/sprint_2022_result.csv', index=False)
sprint_race_res_2023.to_csv('sprint_race_result(processed)/sprint_2023_result.csv', index=False)
sprint_race_res_2024.to_csv('sprint_race_result(processed)/sprint_2024_result.csv', index=False)
sprint_race_res_2025.to_csv('sprint_race_result(processed)/sprint_2025_result.csv', index=False)